# CSCI 544 Group 17 - Step 3: Transformer Fine-Tuning
---
This notebook covers:
1. **DistilBERT** fine-tuning (Person 3)
2. **RoBERTa** fine-tuning (Person 3)
3. **BERTweet** fine-tuning (Person 4)
4. **Metadata ablation** - text only vs text+keyword vs text+keyword+location (Person 4)

**Prerequisites:** Run Step 1/2 notebook first. Processed data should be in `data/` folder on Google Drive.

---
## 0. Setup

Install the HuggingFace Transformers library, mount Google Drive to access the preprocessed data from Step 1/2.

Import all required libraries (PyTorch, sklearn, transformers), and verify GPU availability for model training.

In [ ]:
!pip install transformers datasets -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/CS544-Group17-Project/data/'
INPUT_DIR = '/content/drive/MyDrive/CS544-Group17-Project/input/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


---
## 1. Load Processed Data

Load the preprocessed training and test DataFrames generated by Step 1/2 from Google Drive, containing the `text_transformer` column for model input and metadata columns for ablation experiments.

Note that these data are already preprocessed.

In [ ]:
train_df = pd.read_csv(DATA_DIR + 'train_processed.csv')
test_df = pd.read_csv(DATA_DIR + 'test_processed.csv')

print(f'Train: {train_df.shape}, Test: {test_df.shape}')
print(f'Columns: {list(train_df.columns)}')
train_df.head(3)

Train: (7503, 12), Test: (3263, 9)
Columns: ['id', 'keyword', 'location', 'text', 'target', 'text_len', 'word_count', 'text_clean', 'text_transformer', 'keyword_clean', 'has_keyword', 'has_location']


,id,keyword,location,text,target,text_len,word_count,text_clean,text_transformer,keyword_clean,has_keyword,has_location
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1,69,13,deed reason earthquake may allah forgive u,Our Deeds are the Reason of this earthquake Ma...,NaN,0,0
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1,38,7,forest fire near la ronge sask canada,Forest fire near La Ronge Sask. Canada,NaN,0,0
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1,133,22,resident asked shelter place notified officer ...,All residents asked to 'shelter in place' are ...,NaN,0,0


---
## 2. Dataset & Training Utilities

Define the shared infrastructure used by all experiments in this notebook:
*   **PyTorch Dataset class**: Tokenizes tweet text into model-ready input sequences
*   **Training function**: Handles forward pass / loss / backpropagation / gradient clipping / learning rate scheduling per epoch
*   **Evaluation function**: Computes F1 / Precision / Recall / Accuracy on validation data,
*   **Unified experiment runner**: orchestrates 5-fold stratified cross-validation for any given model and text input.

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler):
  """A complete epoch: Feed data by batches -> Calculate loss -> Backpropagation gradient -> Gradient descent -> Weight update -> LR update"""
    model.train()
    total_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)


def evaluate(model, dataloader):
  """Predict and get F1, Precision, Recall, Accuracy"""
    model.eval()
    preds, true_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            pred = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            preds.extend(pred)
            true_labels.extend(batch['label'].numpy())
    return {
        'F1': f1_score(true_labels, preds),
        'Precision': precision_score(true_labels, preds),
        'Recall': recall_score(true_labels, preds),
        'Accuracy': accuracy_score(true_labels, preds),
        'preds': preds,
        'true': true_labels
    }

In [ ]:
def run_experiment(model_name, texts, labels, epochs=3, batch_size=16, lr=2e-5):
    """
    Run 5-fold CV for a given model and text input.
    This function automates the full pipeline for a single experiment:
    1. Load the tokenizer for the given pretrained model
    2. Split data into 5 stratified folds
    3. For each fold:
       - Initialize a fresh model with a randomly initialized classification head
       - Create train/val DataLoaders from the TweetDataset class
       - Configure AdamW optimizer with weight decay and linear warmup scheduler
       - Train for the specified number of epochs, evaluating after each epoch
       - Record the best F1 score achieved across all epochs for this fold
       - Free GPU memory before moving to the next fold
    4. Compute and print the 5-fold average and standard deviation of F1, Precision, Recall, and Accuracy

    """
    print(f'\n{"="*60}')
    print(f'Model: {model_name}')
    print(f'Samples: {len(texts)}, Epochs: {epochs}, LR: {lr}')
    print(f'{"="*60}')

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels)):
        print(f'\n--- Fold {fold+1}/5 ---')
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

        train_ds = TweetDataset(texts[train_idx], labels[train_idx], tokenizer)
        val_ds = TweetDataset(texts[val_idx], labels[val_idx], tokenizer)
        train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_dl = DataLoader(val_ds, batch_size=batch_size)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = len(train_dl) * epochs
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

        best_f1 = 0
        for epoch in range(epochs):
            loss = train_epoch(model, train_dl, optimizer, scheduler)
            result = evaluate(model, val_dl)
            print(f'  Epoch {epoch+1}: loss={loss:.4f}, F1={result["F1"]:.4f}, P={result["Precision"]:.4f}, R={result["Recall"]:.4f}')
            if result['F1'] > best_f1:
                best_f1 = result['F1']
                best_result = result

        fold_results.append(best_result)
        print(f'  Best F1: {best_f1:.4f}')
        del model
        torch.cuda.empty_cache()

    avg = {k: np.mean([r[k] for r in fold_results]) for k in ['F1','Precision','Recall','Accuracy']}
    std = {k: np.std([r[k] for r in fold_results]) for k in ['F1','Precision','Recall','Accuracy']}
    print(f'\n>>> 5-Fold Average: F1={avg["F1"]:.4f}±{std["F1"]:.4f}, P={avg["Precision"]:.4f}, R={avg["Recall"]:.4f}, Acc={avg["Accuracy"]:.4f}')
    return avg, fold_results

---
## 3. Prepare Text Inputs

Construct three versions of input text for the ablation study:

* text only
* text + keyword (Tweets appended with keyword via [SEP], check Step1Step2 file for detailed description)
* text + keyword + location (tweet appended with both metadata fields)

These 3 variants allow us to isolate the contribution of each metadata field to classification performance.

In [ ]:
texts = train_df['text_transformer'].fillna('').values
labels = train_df['target'].values

# Metadata columns for ablation
keywords = train_df['keyword_clean'].fillna('none').values
locations = train_df['location'].fillna('none').values

# Ablation variants
texts_only = texts
texts_kw = np.array([f'{t} [SEP] {k}' for t, k in zip(texts, keywords)])
texts_kw_loc = np.array([f'{t} [SEP] {k} [SEP] {l}' for t, k, l in zip(texts, keywords, locations)])

print(f'Text only:     {texts_only[0][:80]}...')
print(f'Text+KW:       {texts_kw[0][:80]}...')
print(f'Text+KW+Loc:   {texts_kw_loc[0][:80]}...')

Text only:     Our Deeds are the Reason of this earthquake May ALLAH Forgive us all...
Text+KW:       Our Deeds are the Reason of this earthquake May ALLAH Forgive us all [SEP] none...
Text+KW+Loc:   Our Deeds are the Reason of this earthquake May ALLAH Forgive us all [SEP] none ...


---
## 4. DistilBERT

Fine-tune distilbert-base-uncased on text-only input with 5-fold stratified CV (3 epochs, lr=2e-5, batch size 16).

DistilBERT is a compressed version of BERT with 40% fewer parameters, serving as a lightweight transformer baseline.

In [ ]:
distilbert_avg, distilbert_folds = run_experiment(
    model_name='distilbert-base-uncased',
    texts=texts_only,
    labels=labels,
    epochs=3, batch_size=16, lr=2e-5
)


Model: distilbert-base-uncased
Samples: 7503, Epochs: 3, LR: 2e-05


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1: loss=0.4689, F1=0.7872, P=0.8236, R=0.7539
  Epoch 2: loss=0.3246, F1=0.7889, P=0.8217, R=0.7586
  Epoch 3: loss=0.2519, F1=0.7915, P=0.8274, R=0.7586
  Best F1: 0.7915

--- Fold 5/5 ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1: loss=0.4660, F1=0.8194, P=0.8320, R=0.8072
  Epoch 2: loss=0.3311, F1=0.8245, P=0.8737, R=0.7806
  Epoch 3: loss=0.2540, F1=0.8277, P=0.8310, R=0.8245
  Best F1: 0.8277

>>> 5-Fold Average: F1=0.8081±0.0144, P=0.8370, R=0.7816, Acc=0.8423


---
## 5. RoBERTa

Fine-tune roberta-base on text-only input under the same protocol.

RoBERTa improves on BERT through longer training, larger batches, and removal of the Next Sentence Prediction objective.

In [ ]:
roberta_avg, roberta_folds = run_experiment(
    model_name='roberta-base',
    texts=texts_only,
    labels=labels,
    epochs=3, batch_size=16, lr=2e-5
)


Model: roberta-base
Samples: 7503, Epochs: 3, LR: 2e-05


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1: loss=0.4796, F1=0.7594, P=0.8887, R=0.6630
  Epoch 2: loss=0.3461, F1=0.7945, P=0.8282, R=0.7633
  Epoch 3: loss=0.2647, F1=0.7907, P=0.8062, R=0.7759
  Best F1: 0.7945

--- Fold 5/5 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1: loss=0.4891, F1=0.7911, P=0.9191, R=0.6944
  Epoch 2: loss=0.3598, F1=0.8222, P=0.8571, R=0.7900
  Epoch 3: loss=0.2736, F1=0.8206, P=0.8312, R=0.8103
  Best F1: 0.8222

>>> 5-Fold Average: F1=0.8046±0.0110, P=0.8399, R=0.7722, Acc=0.8405


---
## 6. BERTweet - Text Only

Fine-tune vinai/bertweet-base on text-only input under the same protocol.

BERTweet is pretrained on 850 million English tweets using a RoBERTa-style objective, making it specifically suited for informal social media text with hashtags, abbreviations, and non-standard spelling.

In [ ]:
bertweet_avg, bertweet_folds = run_experiment(
    model_name='vinai/bertweet-base',
    texts=texts_only,
    labels=labels,
    epochs=3, batch_size=16, lr=2e-5
)


Model: vinai/bertweet-base
Samples: 7503, Epochs: 3, LR: 2e-05


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

  Epoch 1: loss=0.4950, F1=0.7789, P=0.8513, R=0.7179
  Epoch 2: loss=0.3488, F1=0.7908, P=0.8296, R=0.7555
  Epoch 3: loss=0.2761, F1=0.7944, P=0.8319, R=0.7602
  Best F1: 0.7944

--- Fold 5/5 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

  Epoch 1: loss=0.4941, F1=0.8175, P=0.8003, R=0.8354
  Epoch 2: loss=0.3525, F1=0.8253, P=0.8443, R=0.8072
  Epoch 3: loss=0.2736, F1=0.8221, P=0.8293, R=0.8150
  Best F1: 0.8253

>>> 5-Fold Average: F1=0.8068±0.0138, P=0.8467, R=0.7716, Acc=0.8431


---
## 7. BERTweet Ablation: Metadata

Re-run BERTweet fine-tuning with two additional input configurations to test whether metadata improves classification: text + keyword (F1 = 0.8056, slightly worse than text-only) and text + keyword + location.

In [ ]:
# Ablation 1: text + keyword
bertweet_kw_avg, _ = run_experiment(
    model_name='vinai/bertweet-base',
    texts=texts_kw,
    labels=labels,
    epochs=3, batch_size=16, lr=2e-5
)

In [ ]:
# Ablation 2: text + keyword + location
bertweet_kw_loc_avg, _ = run_experiment(
    model_name='vinai/bertweet-base',
    texts=texts_kw_loc,
    labels=labels,
    epochs=3, batch_size=16, lr=2e-5
)


Model: vinai/bertweet-base
Samples: 7503, Epochs: 3, LR: 2e-05


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0



--- Fold 1/5 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

...

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

  Epoch 1: loss=0.4813, F1=0.7891, P=0.8079, R=0.7712
  Epoch 2: loss=0.3456, F1=0.7955, P=0.8179, R=0.7743
  Epoch 3: loss=0.2756, F1=0.7936, P=0.8071, R=0.7806
  Best F1: 0.7955

--- Fold 5/5 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

  Epoch 1: loss=0.4931, F1=0.8128, P=0.8122, R=0.8135
  Epoch 2: loss=0.3684, F1=0.8182, P=0.9034, R=0.7476
  Epoch 3: loss=0.3054, F1=0.8193, P=0.8251, R=0.8135
  Best F1: 0.8193

>>> 5-Fold Average: F1=0.8056±0.0110, P=0.8315, R=0.7816, Acc=0.8397


---
## 8. Results Comparison

Consolidate the 5-fold CV results of all five experiments (DistilBERT, RoBERTa, BERTweet text-only, BERTweet + keyword, BERTweet + keyword + location) into a single summary table for direct comparison of F1, Precision, Recall, and Accuracy across models and input configurations.

In [ ]:
all_results = {
    'DistilBERT (text)': distilbert_avg,
    'RoBERTa (text)': roberta_avg,
    'BERTweet (text)': bertweet_avg,
    'BERTweet (text+kw)': bertweet_kw_avg,
    'BERTweet (text+kw+loc)': bertweet_kw_loc_avg,
}

print(f'{"="*70}')
print(f'{"Model":<30s} {"F1":>8s} {"Prec":>8s} {"Recall":>8s} {"Acc":>8s}')
print(f'{"-"*70}')
for name, r in all_results.items():
    print(f'{name:<30s} {r["F1"]:8.4f} {r["Precision"]:8.4f} {r["Recall"]:8.4f} {r["Accuracy"]:8.4f}')
print(f'{"-"*70}')

---
## 9. Generate Kaggle Submission (Best Model)

Retrain the best-performing model (BERTweet) on the full training set (all 7,503 samples) for 3 epochs without cross-validation to maximize training data usage, then generate predictions on the 3,263 unlabeled test tweets. The output is saved as `submission_transformer.csv`. As with the baseline submission in Step 2, this file is ready for Kaggle upload should we choose to benchmark our transformer model on the public leaderboard.

Again, If we are to join the Kaggle competition, use the output `submission_transformer.csv` in data folder.

This section's out put is **NOT included** in our report.

In [ ]:
# Retrain best model on full training data and predict test set
# Change model_name below to whichever performed best above
BEST_MODEL = 'vinai/bertweet-base'  # <- update if needed

tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL, num_labels=2).to(device)

# Train on all data
full_ds = TweetDataset(texts_only, labels, tokenizer)
full_dl = DataLoader(full_ds, batch_size=16, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(full_dl) * 3
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

for epoch in range(3):
    loss = train_epoch(model, full_dl, optimizer, scheduler)
    print(f'Epoch {epoch+1}: loss={loss:.4f}')

In [ ]:
# Predict test set
test_texts = test_df['text_transformer'].fillna('').values
test_ds = TweetDataset(test_texts, np.zeros(len(test_texts), dtype=int), tokenizer)
test_dl = DataLoader(test_ds, batch_size=16)

model.eval()
test_preds = []
with torch.no_grad():
    for batch in test_dl:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pred = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        test_preds.extend(pred)

submission = pd.DataFrame({'id': test_df['id'], 'target': test_preds})
submission.to_csv('submission_transformer.csv', index=False)
print(f'Submission saved! Shape: {submission.shape}')
print(f'Predicted: 0={sum(np.array(test_preds)==0)}, 1={sum(np.array(test_preds)==1)}')

Submission saved! Shape: (3263, 2)
Predicted: 0=1949, 1=1314


---
## 10. Summary

This section summarizes the full progress of Step 3 and compares results against the baselines established in Step 1/2.

**What has been completed (Step 3):**
- Fine-tuned three pretrained transformer models (DistilBERT, RoBERTa, BERTweet) with 5-fold stratified CV
- Conducted metadata ablation on BERTweet (text only vs. text + keyword vs. text + keyword + location)
- Compared all transformer results in a unified summary table
- Generated a transformer-based Kaggle submission file using the best model (BERTweet)

**Key findings:**
- All three transformers significantly outperform the best TF-IDF baseline (LR F1 = 0.7484), improving F1 by ~6 points
- DistilBERT (F1 = 0.8081), RoBERTa (F1 = 0.8046), and BERTweet (F1 = 0.8068) perform within ±0.01 of each other, indicating that the shift from sparse to contextual representations matters more than the specific transformer architecture
- Appending keyword metadata via [SEP] concatenation does not improve BERTweet's F1 (0.8068 → 0.8056), suggesting the encoder already captures this information from the tweet text
- BERTweet achieves the highest precision (0.8467) among all models; DistilBERT achieves the highest F1 despite being 40% smaller

**Next steps:**
- Threshold tuning on validation predictions to optimize F1 decision boundary (zero GPU cost)
- Soft-voting ensemble averaging the best baseline (LR) and best transformer (BERTweet) probabilities
- Class weight adjustment or EDA augmentation to improve recall on the minority class
- Deeper error analysis comparing transformer failure cases against baseline failure cases